# MATEN/T MiniAI v1.1 — Full 100M pretraining and optional LoRA

**BY MATEN/T · M/ License v1.0**

This notebook trains a 100,062,768-parameter model with a 6,144-token maximum context. A normal pretraining job has no artificial 100-step limit. The 9.4M-row generated dataset is synthetic; high-quality natural text is recommended for real capability. **Use a GPU runtime** and persistent Google Drive space. The notebook does not download trained weights.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))


## 1. Fetch the source and install dependencies

This notebook uses the official GitHub repository and requires enough disk space for generated training data. Do not run a second Git clone over an existing project.


In [ ]:
from pathlib import Path
import subprocess,sys
PROJECT=Path('/content/MATEN-T-MiniAI')
if not (PROJECT/'train_full.py').exists():
    subprocess.run(['git','clone','https://github.com/opentaiofficial/MATEN-T-MiniAI.git',str(PROJECT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt')],check=True)


## 2. Prepare the corpus on persistent Drive storage

The next step creates the 94 synthetic shards if absent. Generating 9.4M records takes time and storage. For better results, substitute your own licensed real-text corpus instead. Do not store personal/private texts without permission.


In [ ]:
DATA=Path('/content/drive/MyDrive/MATEN-T-MiniAI-v1.1/data')
DATA.mkdir(parents=True,exist_ok=True)
from importlib.machinery import SourceFileLoader
sys.path.insert(0,str(PROJECT))
from build_datasets import FILES
from tools.verify_datasets import verify
import json
manifest=DATA/'manifest.json'
ready=False
if manifest.is_file():
    metadata=json.loads(manifest.read_text(encoding='utf-8'))
    ready=(metadata.get('total_records')==9_400_000 and len(metadata.get('domains',{}))==94
           and all((DATA/(domain+'.jsonl')).is_file() for domain in FILES))
if not ready:
    subprocess.run([sys.executable,str(PROJECT/'build_datasets.py'),'--out',str(DATA),'--count','100000'],cwd=PROJECT,check=True)
assert verify(DATA), 'Data integrity validation failed; fix before training.'


## 3. Run complete pretraining with automatic resume and SafeTensors export

Persist the corpus, tokenizer and checkpoints on Drive. The default training window here is 512 tokens to reduce memory usage; changing `SEQUENCE_LENGTH` to 6144 requires a much larger GPU. **No --max-steps parameter** is present.


In [ ]:
TOKEN=Path('/content/drive/MyDrive/MATEN-T-MiniAI-v1.1/tokenizer/tokenizer.model')
OUT=Path('/content/drive/MyDrive/MATEN-T-MiniAI-v1.1/pretrain_checkpoints')
TOKEN.parent.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)
SEQUENCE_LENGTH=512  # Increase only if GPU VRAM permits.
EPOCHS=1
subprocess.run([sys.executable,str(PROJECT/'train_full.py'),
               '--corpus',str(DATA),'--tokenizer',str(TOKEN),
               '--output',str(OUT),'--device','cuda','--precision','fp16',
               '--epochs',str(EPOCHS),'--sequence-length',str(SEQUENCE_LENGTH),
               '--batch-size','1','--grad-accum','8','--save-every','500'],
               cwd=PROJECT,check=True)
print('Export:', OUT/'release/model.safetensors')


## 4. Optional LoRA fine-tuning (after pretraining)

Only run this cell after obtaining a pretrained checkpoint. The synthetic instruction/response data can exercise the fine-tuning pipeline, but real human-reviewed licensed dialogs are recommended.


In [ ]:
RUN_LORA=False
if RUN_LORA:
    subprocess.run([sys.executable,str(PROJECT/'finetune_lora.py'),
        '--base',str(OUT/'last.pt'),'--data',str(DATA),
        '--tokenizer',str(TOKEN),'--output',str(OUT/'lora'),
        '--device','cuda','--precision','fp16'],cwd=PROJECT,check=True)


## 5. Check release files

The SafeTensors release contains model weights, config and SentencePiece tokenizer. Weight files are not published to GitHub automatically.


In [ ]:
from pathlib import Path
folder=OUT/'release'
print('Release ready:',(folder/'model.safetensors').exists())
for name in ['model.safetensors','model.sha256','config.json','tokenizer.model','tokenizer_config.json','generation_config.json']:
    print(name, (folder/name).exists())
